In [ ]:
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

In [ ]:
model_name = 'qwen2vl'
prompt_type = 'zeroshot'

In [ ]:
dataset = 'type1'
image_type = 'original'
device = "cuda:1"

In [ ]:
qas = pd.read_json(f'../{dataset}/qa.json')
qas

In [ ]:
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values

In [ ]:
image_base_path = f'../{dataset}/{image_type}/'

In [ ]:
all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

index_to_image

In [ ]:
message_template = [
    {
        "role": "user",
        "content": []
    }
]

In [ ]:
def get_message(image_index, prompt, question):
    images = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        images.append({
            "type": "image",
            "image": image_path
        })
    message = message_template.copy()
    message[0]['content'] = images
    message[0]['content'].append({
        "type": "text",
        "text": prompt.format(question)
    })
    return message

In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map=device,
    cache_dir='/media/vivek/drive/multichartqa/models_cache'
)
model.eval()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

In [ ]:
model_responses = []

In [ ]:
def run_model():
    for i, question in enumerate(tqdm(questions)):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        messages = get_message(image_index, prompt, question)
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(device)
        
        generated_ids = model.generate(
            **inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': output_text[0].strip()})

In [ ]:
run_model()

In [ ]:
# saving the model responses
os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

model_responses_df = pd.DataFrame(model_responses)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')